<a href="https://colab.research.google.com/github/25WH1A0538/NASSCOM/blob/main/WEEK_3_U8_Data_Cleaning_Part1_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.3


In [ ]:
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [ ]:
df = raw.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [ ]:
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [ ]:

print("Duplicate rows:", df.duplicated().sum())

print("Missing per column:")
print(df.isna().sum())

# 3. Problems I can see:
# - There are duplicate rows.
# - Some columns have missing values.
# - The 'spend' column is stored as text instead of a numeric type.

Duplicate rows: 1
Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [ ]:
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')
df['age']   = df['age'].replace(-1, np.nan)

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

Missing after unmasking:
age      2
spend    1
dtype: int64


In [ ]:
ex = raw.copy()

ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)

ex_drop = ex.dropna()

ex_impute = ex.copy()
ex_impute['age'] = ex_impute['age'].fillna(ex_impute['age'].median())
ex_impute['spend'] = ex_impute['spend'].fillna(ex_impute['spend'].median())

print("Original rows:", len(ex))
print("Rows after dropna:", len(ex_drop))
print("Rows after median imputation:", len(ex_impute))
print("Rows lost by dropping:", len(ex) - len(ex_drop))

Original rows: 8
Rows after dropna: 5
Rows after median imputation: 8
Rows lost by dropping: 3


In [ ]:
print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')


Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['city'] = df['city'].astype('string')
print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [ ]:
ex = raw.copy()
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'], errors='coerce')
ex = ex.drop_duplicates()
print(ex.dtypes)
print(ex.shape)

id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object
(7, 6)


In [ ]:
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')
outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=112.6  Q3=187.6  IQR=75.0
Normal range: 0.1 to 300.2

Outlier rows:
  name    spend
5  Fin  99000.0


In [ ]:
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5      120.5000
1   Bo     80.0       80.0000
2   Cy    200.2      200.2000
3   Di      NaN           NaN
4  Eve    150.0      150.0000
5  Fin  99000.0      300.1875
6  Gus    110.0      110.0000


In [ ]:
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
outliers = df[(df['age'] < lower_bound) | (df['age'] > upper_bound)]
print("Outlier rows:")
print(outliers)

Q1: 29.0
Q3: 38.0
IQR: 9.0
Lower bound: 15.5
Upper bound: 51.5
Outlier rows:
Empty DataFrame
Columns: [id, name, age, city, spend, date, spend_capped]
Index: []


In [ ]:
print(df['city'].value_counts())


city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


In [ ]:
s = df['city'].astype('string')
s = s.str.strip()
s = s.str.lower()
s = s.str.replace('.', '', regex=False)
s = s.replace({'bombay': 'mumbai'})
df['city'] = s
print(df['city'].value_counts())

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [ ]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')
messy = messy.str.strip().str.lower()
messy = messy.replace('n.y.', 'new york')
print(messy.value_counts())

london      3
new york    3
Name: count, dtype: Int64


In [ ]:
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (7, 6)
Missing values: 3
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,NaN,2024-01-08
4,5,Eve,NaN,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
